#Imports

In [ ]:
import json
import os
import pandas as pd
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from tqdm import tqdm
from sklearn.metrics import classification_report
from transformers import LayoutLMv3Processor, LayoutLMv3ForTokenClassification
from PIL import Image
from datasets import Dataset
from pathlib import Path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!git clone https://github.com/rossumai/docile.git

Cloning into 'docile'...
remote: Enumerating objects: 1032, done.
remote: Counting objects: 100% (353/353), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 1032 (delta 246), reused 225 (delta 201), pack-reused 679 (from 1)
Receiving objects: 100% (1032/1032), 976.26 KiB | 7.02 MiB/s, done.
Resolving deltas: 100% (626/626), done.


In [ ]:
%cd docile

[Errno 2] No such file or directory: 'docile'
/content


In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers datasets seqeval evaluate pdf2image pyarrow scikit-learn accelerate pillow
!apt-get install -y poppler-utils

Looking in indexes: https://download.pytorch.org/whl/cu121
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.10).
0 upgraded, 0 newly installed, 0 to remove and 38 not upgraded.


Setting the file paths

In [ ]:
base_path  = "/content/drive/MyDrive/AI_PROJECT_FOLDER"
val_json_path = os.path.join(base_path,'val.json')
annotations_path  = os.path.join(base_path,'annotations')
TRAIN_JSON = os.path.join(base_path,'train.json')
pdf_path = os.path.join(base_path,'pdfs')
OUTPUT_DIR = Path("/content/drive/MyDrive/roberta_docile_checkpoints")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Extracting the validation file names (for zero shot training)
* locating the validation annotated files




In [ ]:
with open(val_json_path, "r") as f:
    val_filenames = json.load(f)
val_annotation_files = []
missing = []
for fname in val_filenames:
    p = os.path.join(annotations_path,fname)
    if os.path.exists(p):
        val_annotation_files.append(p)
    else:
        p_alt = os.path.join(annotations_path,(fname if fname.endswith(".json") else f"{fname}.json"))
        if os.path.exists(p_alt):
            val_annotation_files.append(p_alt)
        else:
            matches = list(annotations_path.glob(f"**/{Path(fname).name}"))
            if matches:
                val_annotation_files.append(matches[0])
            else:
                missing.append(fname)

print(f"Found {len(val_annotation_files)} validation annotated files, missing {len(missing)}")
if missing:
    print("Missing examples (first 10):", missing[:10])


Found 500 validation annotated files, missing 0


# Extracting the text and their labels from the validation files

In [ ]:
# opens a single annotated file and loads it into dictionary 'data'
def extract_text_and_label(ann_path):
    with open(ann_path, 'r') as f:
        data = json.load(f)
    texts = []
    labels = []

    for item in data.get("field_extractions", []):
        text = item.get("text", None)
        label = item.get("fieldtype", None)
        if text is None or label is None:
            continue
        texts.append(text)
        labels.append(label)
    return texts, labels

all_texts = []
all_labels = []

for file in val_annotation_files:
    texts, labels = extract_text_and_label(file)
    all_texts.extend(texts)
    all_labels.extend(labels)

print(f"Total texts: {len(all_texts)}, Total labels: {len(all_labels)}")
print("Sample texts & labels:", list(zip(all_texts, all_labels))[:5])


#Zero-Shot on RoBERTa-Base

Tokenizing the text (Building the Dataset for RoBERTa)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")
unique_labels = sorted(list(set(all_labels)))
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}
class RoBERTaValDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = [label2id[label] for label in labels]
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item["labels"] = torch.tensor(label)
        return item

val_dataset = RoBERTaValDataset(all_texts, all_labels, tokenizer, max_len=128)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

print(f"Validation dataset size: {len(val_dataset)}")
print(f"Unique dataset size: {len(unique_labels)}")



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Validation dataset size: 5862
Unique dataset size: 33


Extracting the performance metrics for baseline evaluation

In [ ]:
device = torch.device("cpu")
model = AutoModelForSequenceClassification.from_pretrained("roberta-base",num_labels=len(unique_labels)).to(device)

model.eval()

all_preds = []
all_true = []

for batch in tqdm(val_loader):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=-1)
    all_preds.extend(preds.cpu().tolist())
    all_true.extend(labels.cpu().tolist())

prec = precision_score(all_true, all_preds, average="micro")
rec = recall_score(all_true, all_preds, average="micro")
f1 = f1_score(all_true, all_preds, average="micro")
acc = accuracy_score(all_true, all_preds)
print("\n")
print(f"Baseline validation accuracy: {acc:.4f}")
print(f"Baseline validation f1: {f1:.4f}")
print(f"Baseline validation recall: {rec:.4f}")
print(f"Baseline validation precision: {prec:.4f}")

# -----------------------------------------------------------------------------------

#TRAINING + VALIDATION

Extracting training file names

In [ ]:
with open(TRAIN_JSON, "r") as f:
    train_filenames = json.load(f)

train_annotation_files = []
missing = []
for fname in train_filenames:
    p = os.path.join(annotations_path,fname)
    if os.path.exists(p):
        train_annotation_files.append(p)
    else:
        p_alt = os.path.join(annotations_path,(fname if fname.endswith(".json") else f"{fname}.json"))
        if os.path.join(p_alt):
            train_annotation_files.append(p_alt)
        else:
            matches = list(annotations_path.glob(f"**/{Path(fname).name}"))
            if matches:
                train_annotation_files.append(matches[0])
            else:
                missing.append(fname)

print(f"Found {len(train_annotation_files)} annotation files, missing {len(missing)}")
if missing:
    print("Missing examples (first 10):", missing[:10])


Found 5180 annotation files, missing 0


Extracting text and labels from the training and validation dataset

In [ ]:
from pathlib import Path
import json

def load_split_names(split_json_path):
    with open(split_json_path, "r") as f:
        names = json.load(f)
    return names

def find_annotation_file(annotations_dir: Path, name: str):
    cand = os.path.join(annotations_dir,name)
    cand2 = os.path.join(annotations_dir,(name if name.endswith(".json") else f"{name}.json"))
    if os.path.exists(cand): return cand
    if os.path.exists(cand2): return cand2
    matches = list(annotations_dir.glob(f"**/{Path(name).name}"))
    if matches: return matches[0]
    return None

def build_text_label_lists(split_json_path, annotations_dir: Path):
    names = load_split_names(split_json_path)
    texts = []
    labels = []
    filenames = []
    missing = []
    for nm in names:
        ann_path = find_annotation_file(annotations_dir, nm)
        if ann_path is None:
            missing.append(nm)
            continue
        with open(ann_path, "r") as f:
            data = json.load(f)
        for item in data.get("field_extractions", []):
            text = item.get("text", "")
            label = item.get("fieldtype", None)
            if not text or label is None:
                continue
            texts.append(text.strip())
            labels.append(label)
            # filenames.append(ann_path.name)
    if missing:
        print(f"Warning: {len(missing)} filenames from {split_json_path} not found (showing first 10): {missing[:10]}")
    return texts, labels, filenames

train_texts, train_labels, train_files = build_text_label_lists(TRAIN_JSON, annotations_path)
val_texts, val_labels, val_files = build_text_label_lists(val_json_path, annotations_path)

print(f"Train samples: {len(train_texts)}  Val samples: {len(val_texts)}")

Train samples: 65651  Val samples: 5862


### Saving the text and labels to a file

In [ ]:
TRAIN_CACHE = Path("/content/drive/MyDrive/AI_PROJECT_FOLDER/train_extracted.json")
VAL_CACHE = Path("/content/drive/MyDrive/AI_PROJECT_FOLDER/val_extracted.json")

In [ ]:
def save_extracted_data(texts, labels, filenames, save_path: Path):
    data = {"texts": texts, "labels": labels, "filenames": filenames}
    with open(save_path, "w") as f:
        json.dump(data, f)
    print(f"✅ Saved extracted data to: {save_path}")

save_extracted_data(train_texts, train_labels, train_files, TRAIN_CACHE)
save_extracted_data(val_texts, val_labels, val_files, VAL_CACHE)

### loading the data from the file

In [ ]:
with open(TRAIN_CACHE, "r") as f:
    train_data = json.load(f)
with open(VAL_CACHE, "r") as f:
    val_data = json.load(f)

train_texts = train_data["texts"]
train_labels = train_data["labels"]
val_texts = val_data["texts"]
val_labels = val_data["labels"]

Exploring the number of labels

In [ ]:
unique_labels = sorted(list(set(train_labels + val_labels)))
label2id = {lab: idx for idx, lab in enumerate(unique_labels)}
id2label = {idx: lab for lab, idx in label2id.items()}

train_label_ids = [label2id[l] for l in train_labels]
val_label_ids = [label2id[l] for l in val_labels]

print(f"Number of classes: {len(unique_labels)}")
print("Example labels:", unique_labels[:10])

Number of classes: 36
Example labels: ['account_num', 'amount_due', 'amount_paid', 'amount_total_gross', 'amount_total_net', 'amount_total_tax', 'bank_num', 'bic', 'currency_code_amount_due', 'customer_billing_address']


Tokenizing the texts

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

train_ds = Dataset.from_dict({"text": train_texts, "label": train_label_ids})
val_ds   = Dataset.from_dict({"text": val_texts,   "label": val_label_ids})

MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

max_length = 128
def tokenize_batch(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=max_length)

train_ds = train_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
val_ds   = val_ds.map(tokenize_batch, batched=True, remove_columns=["text"])

train_ds.set_format(type="torch")
val_ds.set_format(type="torch")

print(train_ds[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/65651 [00:00<?, ? examples/s]

Map:   0%|          | 0/5862 [00:00<?, ? examples/s]

{'label': tensor(21), 'input_ids': tensor([    0, 27204,   541, 13872,   176,     2,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1, 

#RoBERTa Base Full Fine Tuning

Setting the parameters

In [ ]:
import numpy as np
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

num_labels = len(unique_labels)
model = AutoModelForSequenceClassification.from_pretrained(latest_ckpt)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    prec_macro = precision_score(labels, preds, average="macro", zero_division=0)
    prec_micro = precision_score(labels, preds, average="micro", zero_division=0)
    rec_macro = recall_score(labels, preds, average="macro", zero_division=0)
    rec_micro = recall_score(labels, preds, average="micro", zero_division=0)
    f1_macro = f1_score(labels, preds, average="macro", zero_division=0)
    f1_micro = f1_score(labels, preds, average="micro", zero_division=0)
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "prec_macro": prec_macro,
        "prec_micro": prec_micro,
        "rec_macro": rec_macro,
        "rec_micro": rec_micro,
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
    }

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=True,
    logging_steps=100,
    save_total_limit=2,
)

Training the Model

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

### Run the code below to create the epoch vs loss curve
*   after training is done, use the accurate training values to input inside the training_loss and validation_loss lists in the code below


In [ ]:
import matplotlib.pyplot as plt

epochs = [1, 2, 3, 4, 5]
training_loss = [0.577200,0.524700, 0.469800, 0.483100, 0.469300]
validation_loss = [0.770839, 0.767892, 0.775565, 0.747574, 0.750147]

plt.figure(figsize=(7,5))
plt.plot(epochs, training_loss, marker='o', label='Training Loss', linewidth=2)
plt.plot(epochs, validation_loss, marker='o', label='Validation Loss', linewidth=2)

plt.title('Epoch vs Loss', fontsize=14, weight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

plt.show()

#RoBERTa Base Partial Tuning 1

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch

num_labels = len(unique_labels)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

for param in model.roberta.embeddings.parameters():
    param.requires_grad = False

for layer in model.roberta.encoder.layer[:8]:
    for param in layer.parameters():
        param.requires_grad = False

for layer in model.roberta.encoder.layer[8:]:
    for param in layer.parameters():
        param.requires_grad = True

for param in model.classifier.parameters():
    param.requires_grad = True

In [ ]:
optimizer_grouped_parameters = [
    {"params": [p for n, p in model.named_parameters() if "classifier" in n and p.requires_grad], "lr": 5e-5},
    {"params": [p for n, p in model.named_parameters() if "roberta.encoder.layer.8" in n or
                "roberta.encoder.layer.9" in n or
                "roberta.encoder.layer.10" in n or
                "roberta.encoder.layer.11" in n and p.requires_grad], "lr": 1e-5},
]


In [ ]:
from transformers import TrainingArguments
per_device_train_batch_size = 16
num_train_epochs = 3

total_steps = (len(train_ds) // per_device_train_batch_size) * num_train_epochs
warmup_steps = int(0.05 * total_steps)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    prec = precision_score(labels, preds, average="micro", zero_division=0)
    rec = recall_score(labels, preds, average="micro", zero_division=0)
    f1 = f1_score(labels, preds, average="micro", zero_division=0)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=32,
    num_train_epochs=num_train_epochs,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=True,
    logging_steps=100,
    save_total_limit=2,
    warmup_steps=warmup_steps
)

In [ ]:
from torch.optim import AdamW
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
optimizer = AdamW(optimizer_grouped_parameters)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, None)
)

trainer.train()

### Run the code below to create the epoch vs loss curve
*   after training is done, use the accurate training values to input inside the training_loss and validation_loss lists in the code below

In [ ]:
import matplotlib.pyplot as plt

epochs = [1, 2, 3]
training_loss = [0.699900, 0.716300, 0.681700, 0.681700, 0.681700]
validation_loss = [0.793351, 0.740720, 0.729280, 0.729280, 0.729280]

plt.figure(figsize=(7,5))
plt.plot(epochs, training_loss, marker='o', label='Training Loss', linewidth=2)
plt.plot(epochs, validation_loss, marker='o', label='Validation Loss', linewidth=2)

plt.title('Epoch vs Loss', fontsize=14, weight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

plt.show()

#RoBERTa Base Partial Tuning 2

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, get_scheduler
from torch.optim import AdamW
from transformers import DataCollatorWithPadding
import torch, numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

MODEL_NAME = "roberta-base"
num_labels = len(unique_labels)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

for param in model.roberta.embeddings.parameters():
    param.requires_grad = False
for layer in model.roberta.encoder.layer[:6]:
    for param in layer.parameters():
        param.requires_grad = False

optimizer_grouped_parameters = [
    {"params": [p for n, p in model.named_parameters() if p.requires_grad and "classifier" not in n],
     "lr": 3e-5},
    {"params": [p for n, p in model.named_parameters() if "classifier" in n],
     "lr": 5e-5},
]

optimizer = AdamW(optimizer_grouped_parameters)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    prec = precision_score(labels, preds, average="micro", zero_division=0)
    rec = recall_score(labels, preds, average="micro", zero_division=0)
    f1 = f1_score(labels, preds, average="micro", zero_division=0)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    max_grad_norm=1.0,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=True,
    logging_steps=100,
    save_total_limit=2,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, None),
)

trainer.train()

### Run the code below to create the epoch vs loss curve
*   after training is done, use the accurate training values to input inside the training_loss and validation_loss lists in the code below

In [ ]:
import matplotlib.pyplot as plt

epochs = [1, 2, 3, 4, 5]
training_loss = [0.811900, 0.706500, 0.630400, 0.587900, 0.584300]
validation_loss = [0.812105, 0.744605, 0.707240, 0.713727, 0.709117]

plt.figure(figsize=(7,5))
plt.plot(epochs, training_loss, marker='o', label='Training Loss', linewidth=2)
plt.plot(epochs, validation_loss, marker='o', label='Validation Loss', linewidth=2)

plt.title('Epoch vs Loss', fontsize=14, weight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

plt.show()

#Roberta Partial Fine Tuning 3

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, get_scheduler
from torch.optim import AdamW
from transformers import DataCollatorWithPadding
import torch, numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

MODEL_NAME = "roberta-base"
num_labels = len(unique_labels)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

for param in model.roberta.embeddings.parameters():
    param.requires_grad = False
for layer in model.roberta.encoder.layer[:6]:
    for param in layer.parameters():
        param.requires_grad = False

optimizer_grouped_parameters = [
    {"params": [p for n, p in model.named_parameters() if p.requires_grad and "classifier" not in n],
     "lr": 3e-5},
    {"params": [p for n, p in model.named_parameters() if "classifier" in n],
     "lr": 5e-5},
]

optimizer = AdamW(optimizer_grouped_parameters)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    prec_macro = precision_score(labels, preds, average="macro", zero_division=0)
    prec_micro = precision_score(labels, preds, average="micro", zero_division=0)
    rec_macro = recall_score(labels, preds, average="macro", zero_division=0)
    rec_micro = recall_score(labels, preds, average="micro", zero_division=0)
    f1_macro = f1_score(labels, preds, average="macro", zero_division=0)
    f1_micro = f1_score(labels, preds, average="micro", zero_division=0)
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "prec_macro": prec_macro,
        "prec_micro": prec_micro,
        "rec_macro": rec_macro,
        "rec_micro": rec_micro,
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
    }

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,              # base lr (we override via optimizer groups)
    per_device_train_batch_size=16,  # or 32 if GPU allows
    per_device_eval_batch_size=64,
    num_train_epochs=5,
    weight_decay=0.05,
    warmup_ratio=0.1,
    max_grad_norm=1.0,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=True,
    fp16_full_eval=True,
    logging_steps=100,
    save_total_limit=2,
    gradient_accumulation_steps=2,
    report_to="none",
)


data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

num_training_steps = len(train_ds) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps) * training_args.num_train_epochs
lr_scheduler = get_scheduler(
    name="cosine",
    optimizer=optimizer,
    num_warmup_steps=int(0.1 * num_training_steps),
    num_training_steps=num_training_steps,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, lr_scheduler),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, None),
)

trainer.train()

### Run the code below to create the epoch vs loss curve
*   after training is done, use the accurate training values to input inside the training_loss and validation_loss lists in the code below

In [ ]:
import matplotlib.pyplot as plt

epochs = [1, 2, 3, 4, 5]
training_loss = [0.817900, 0.715800, 0.676200, 0.588400, 0.579400]
validation_loss = [0.795121, 0.751800, 0.700635, 0.708756,0.697637 ]

plt.figure(figsize=(7,5))
plt.plot(epochs, training_loss, marker='o', label='Training Loss', linewidth=2)
plt.plot(epochs, validation_loss, marker='o', label='Validation Loss', linewidth=2)

plt.title('Epoch vs Loss', fontsize=14, weight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

plt.show()

# Roberta Base Partial Tuning 4

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, get_scheduler, DataCollatorWithPadding
from torch.optim import AdamW
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import numpy as np
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "roberta-base"
num_labels = len(unique_labels)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

for param in model.roberta.embeddings.parameters():
    param.requires_grad = False
for layer in model.roberta.encoder.layer[:4]:
    for param in layer.parameters():
        param.requires_grad = False

optimizer_grouped_parameters = [
    {
        "params": [p for n, p in model.named_parameters() if p.requires_grad and "classifier" not in n],
        "lr": 4e-5,
    },
    {
        "params": [p for n, p in model.named_parameters() if "classifier" in n],
        "lr": 8e-5,
    },
]

optimizer = AdamW(optimizer_grouped_parameters, weight_decay=0.1)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    prec = precision_score(labels, preds, average="macro", zero_division=0)
    rec = recall_score(labels, preds, average="macro", zero_division=0)
    f1 = f1_score(labels, preds, average="macro", zero_division=0)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=4e-5,
    per_device_train_batch_size=24,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.1,
    warmup_ratio=0.06,
    max_grad_norm=1.0,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=True,
    fp16_full_eval=True,
    logging_steps=100,
    save_total_limit=2,
    gradient_accumulation_steps=1,
    report_to="none",
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

num_training_steps = len(train_ds) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps) * training_args.num_train_epochs

lr_scheduler = get_scheduler(
    name="cosine",
    optimizer=optimizer,
    num_warmup_steps=int(0.06 * num_training_steps),
    num_training_steps=num_training_steps,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, lr_scheduler),
)

trainer.train()
trainer.save_model("/content/drive/MyDrive/AI_PROJECT_FOLDER/final_roberta_model")


### Run the code below to create the epoch vs loss curve
*   after training is done, use the accurate training values to input inside the training_loss and validation_loss lists in the code below

In [ ]:
import matplotlib.pyplot as plt

epochs = [1, 2, 3, 4, 5, 6]
training_loss = [0.821700, 0.700500, 0.632500, 0.544200, 0.526200, 0.508700]
validation_loss = [0.801302, 0.748949, 0.697217, 0.705249, 0.698321, 0.708910]

plt.figure(figsize=(7,5))
plt.plot(epochs, training_loss, marker='o', label='Training Loss', linewidth=2)
plt.plot(epochs, validation_loss, marker='o', label='Validation Loss', linewidth=2)

plt.title('Epoch vs Loss', fontsize=14, weight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

plt.show()

# Roberta Partial Tuning 5

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, get_scheduler, DataCollatorWithPadding
from torch.optim import AdamW
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import numpy as np
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "roberta-base"
num_labels = len(unique_labels)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

for param in model.roberta.embeddings.parameters():
    param.requires_grad = False
for layer in model.roberta.encoder.layer[:3]:
    for param in layer.parameters():
        param.requires_grad = False

optimizer_grouped_parameters = [
    {
        "params": [p for n, p in model.named_parameters() if p.requires_grad and "classifier" not in n],
        "lr": 5e-5,
    },
    {
        "params": [p for n, p in model.named_parameters() if "classifier" in n],
        "lr": 9e-5,
    },
]

optimizer = AdamW(optimizer_grouped_parameters, weight_decay=0.15)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    prec = precision_score(labels, preds, average="micro", zero_division=0)
    rec = recall_score(labels, preds, average="micro", zero_division=0)
    f1 = f1_score(labels, preds, average="micro", zero_division=0)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=24,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.15,
    warmup_ratio=0.08,
    lr_scheduler_type="cosine_with_restarts",
    max_grad_norm=0.9,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=True,
    fp16_full_eval=True,
    logging_steps=50,
    save_total_limit=2,
    gradient_accumulation_steps=2,
    report_to="none",
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

num_training_steps = len(train_ds) // (
    training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
) * training_args.num_train_epochs

lr_scheduler = get_scheduler(
    name="cosine_with_restarts",
    optimizer=optimizer,
    num_warmup_steps=int(0.08 * num_training_steps),
    num_training_steps=num_training_steps,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, lr_scheduler),
)

trainer.train()

### Run the code below to create the epoch vs loss curve
*   after training is done, use the accurate training values to input inside the training_loss and validation_loss lists in the code below

In [ ]:
import matplotlib.pyplot as plt

epochs = [1, 2, 3, 4, 5]
training_loss = [0.817900, 0.715800, 0.676200, 0.588400, 0.579400]
validation_loss = [0.795121, 0.751800, 0.700635, 0.708756,0.697637 ]

plt.figure(figsize=(7,5))
plt.plot(epochs, training_loss, marker='o', label='Training Loss', linewidth=2)
plt.plot(epochs, validation_loss, marker='o', label='Validation Loss', linewidth=2)

plt.title('Epoch vs Loss', fontsize=14, weight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

plt.show()

# Roberta Partial Fine Tuning 6

In [ ]:

from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, get_scheduler, DataCollatorWithPadding
from peft import LoraConfig, get_peft_model
from torch.optim import AdamW
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import numpy as np
import torch
!pip install peft -q

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "roberta-base"
num_labels = len(unique_labels)

base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

for param in base_model.roberta.embeddings.parameters():
    param.requires_grad = False
for layer in base_model.roberta.encoder.layer[:4]:
    for param in layer.parameters():
        param.requires_grad = False

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)

model = get_peft_model(base_model, lora_config).to(device)

optimizer_grouped_parameters = [
    {"params": [p for n, p in model.named_parameters() if p.requires_grad and "classifier" not in n],
     "lr": 4e-5},
    {"params": [p for n, p in model.named_parameters() if "classifier" in n],
     "lr": 8e-5},
]

optimizer = AdamW(optimizer_grouped_parameters, weight_decay=0.1)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    prec = precision_score(labels, preds, average="micro", zero_division=0)
    rec = recall_score(labels, preds, average="micro", zero_division=0)
    f1 = f1_score(labels, preds, average="micro", zero_division=0)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=4e-5,
    per_device_train_batch_size=24,
    per_device_eval_batch_size=64,
    num_train_epochs=5,
    weight_decay=0.1,
    warmup_ratio=0.1,
    max_grad_norm=1.0,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=True,
    fp16_full_eval=True,
    logging_steps=100,
    save_total_limit=2,
    gradient_accumulation_steps=1,
    report_to="none",
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

num_training_steps = len(train_ds) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps) * training_args.num_train_epochs

lr_scheduler = get_scheduler(
    name="cosine",
    optimizer=optimizer,
    num_warmup_steps=int(0.1 * num_training_steps),
    num_training_steps=num_training_steps,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, lr_scheduler),
)

trainer.train()


### Run the code below to create the epoch vs loss curve
*   after training is done, use the accurate training values to input inside the training_loss and validation_loss lists in the code below

In [ ]:
import matplotlib.pyplot as plt

epochs = [1, 2, 3, 4, 5]
training_loss = [0.923000, 0.850600, 0.820500, 0.775000, 0.788700]
validation_loss = [0.892478, 0.845731,0.821288, 0.797454, 0.795793 ]

plt.figure(figsize=(7,5))
plt.plot(epochs, training_loss, marker='o', label='Training Loss', linewidth=2)
plt.plot(epochs, validation_loss, marker='o', label='Validation Loss', linewidth=2)

plt.title('Epoch vs Loss', fontsize=14, weight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

plt.show()